In [15]:
from itertools import product
from pathlib import Path
import contextlib
import io

import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

from backtester import Backtester


DATA_PATH = Path("dev_data_30.csv")
TRAIN_PROP = 0.75
N_SPLITS = 5

LOOKBACKS = [20, 60, 120]
VOL_WINDOWS = [10, 30, 40, 50, 60]
MAX_SHARES_GRID = [50,100]
VOL_FLOOR = 1e-6


In [16]:
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%y")
panel = (
    df.pivot(index="Date", columns="Ticker", values="Open")
    .sort_index()
    .astype(float)
)

unique_dates = panel.index.to_numpy()
split_idx = int(len(unique_dates) * TRAIN_PROP)
train_dates = unique_dates[:split_idx]
test_dates = unique_dates[split_idx:]

print(f"Rows: {panel.shape[0]} dates x {panel.shape[1]} tickers")
print(f"Train: {pd.Timestamp(train_dates[0]).date()} -> {pd.Timestamp(train_dates[-1]).date()} ({len(train_dates)} dates)")
print(f"Test:  {pd.Timestamp(test_dates[0]).date()} -> {pd.Timestamp(test_dates[-1]).date()} ({len(test_dates)} dates)")


Rows: 1006 dates x 30 tickers
Train: 2017-03-15 -> 2020-03-12 (754 dates)
Test:  2020-03-13 -> 2021-03-12 (252 dates)


In [17]:
def build_tsmom_strategy(open_panel, lookback, vol_window, max_shares, vol_floor=VOL_FLOOR):
    log_returns = np.log(open_panel).diff()

    # Every signal at day t uses information only through day t-1.
    trend = log_returns.shift(1).rolling(window=lookback, min_periods=lookback).sum()
    vol = log_returns.shift(1).rolling(window=vol_window, min_periods=vol_window).std()
    vol = vol.clip(lower=vol_floor)

    signal = pd.DataFrame(0, index=open_panel.index, columns=open_panel.columns, dtype=int)
    signal[trend > 0] = 1
    signal[trend < 0] = -1

    # Fixed-share sizing via inverse volatility, normalized around the
    # cross-sectional median so the cap matters without forcing every
    # active name to the maximum size.
    inv_vol = (1.0 / vol).where(signal != 0)
    median_inv_vol = inv_vol.median(axis=1)
    relative_inv_vol = inv_vol.div(median_inv_vol.replace(0, np.nan), axis=0)

    shares = np.ceil((max_shares / 3.0) * relative_inv_vol)
    shares = shares.where(signal != 0, 0).fillna(0)
    shares = shares.clip(lower=0, upper=max_shares).astype(int)

    positions = shares * signal
    actions = positions.diff().fillna(positions).astype(int)

    return {
        "trend": trend,
        "vol": vol,
        "signal": signal,
        "shares": shares,
        "positions": positions,
        "actions": actions,
    }


def run_backtest(open_panel, actions):
    with contextlib.redirect_stdout(io.StringIO()):
        backtester = Backtester(
            open_panel.to_numpy(dtype=float).T,
            actions.to_numpy(dtype=int).T,
        )
        port_values, pnl = backtester.eval_actions()

    if port_values is None:
        return None, np.nan

    return np.asarray(port_values, dtype=float), float(pnl)


def summarize_segment(port_values, start_idx, end_idx, initial_cash=25000.0):
    if port_values is None or start_idx > end_idx:
        return {
            "pnl": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "start_value": np.nan,
            "end_value": np.nan,
        }

    prev_value = initial_cash if start_idx == 0 else float(port_values[start_idx - 1])
    segment_values = np.asarray(port_values[start_idx : end_idx + 1], dtype=float)
    pnl = float(segment_values[-1] - prev_value)

    perf_values = np.r_[prev_value, segment_values]
    returns = pd.Series(perf_values).pct_change().dropna()
    if returns.std(ddof=0) > 0:
        sharpe = float(np.sqrt(252) * returns.mean() / returns.std(ddof=0))
    else:
        sharpe = np.nan

    running_max = np.maximum.accumulate(perf_values)
    drawdowns = perf_values / running_max - 1.0
    max_drawdown = float(drawdowns.min())

    return {
        "pnl": pnl,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
        "start_value": prev_value,
        "end_value": float(segment_values[-1]),
    }


def evaluate_params(open_panel, train_date_array, lookback, vol_window, max_shares):
    strategy = build_tsmom_strategy(open_panel, lookback, vol_window, max_shares)
    splitter = TimeSeriesSplit(n_splits=N_SPLITS)
    warmup = max(lookback, vol_window) + 1
    fold_rows = []

    for fold, (fold_train_idx, fold_val_idx) in enumerate(splitter.split(train_date_array), start=1):
        val_dates = train_date_array[fold_val_idx]
        slice_start = max(0, fold_val_idx[0] - warmup)
        slice_dates = train_date_array[slice_start : fold_val_idx[-1] + 1]

        price_slice = open_panel.loc[slice_dates]
        action_slice = strategy["actions"].loc[slice_dates]
        port_values, _ = run_backtest(price_slice, action_slice)

        if port_values is None:
            fold_rows.append(
                {
                    "fold": fold,
                    "train_start": pd.Timestamp(train_date_array[fold_train_idx[0]]).date(),
                    "train_end": pd.Timestamp(train_date_array[fold_train_idx[-1]]).date(),
                    "val_start": pd.Timestamp(val_dates[0]).date(),
                    "val_end": pd.Timestamp(val_dates[-1]).date(),
                    "pnl": np.nan,
                    "sharpe": np.nan,
                    "max_drawdown": np.nan,
                    "turnover": np.nan,
                    "failed": True,
                }
            )
            continue

        val_start_local = fold_val_idx[0] - slice_start
        val_end_local = len(slice_dates) - 1
        metrics = summarize_segment(port_values, val_start_local, val_end_local)
        turnover = int(strategy["actions"].loc[val_dates].abs().to_numpy().sum())

        fold_rows.append(
            {
                "fold": fold,
                "train_start": pd.Timestamp(train_date_array[fold_train_idx[0]]).date(),
                "train_end": pd.Timestamp(train_date_array[fold_train_idx[-1]]).date(),
                "val_start": pd.Timestamp(val_dates[0]).date(),
                "val_end": pd.Timestamp(val_dates[-1]).date(),
                "pnl": metrics["pnl"],
                "sharpe": metrics["sharpe"],
                "max_drawdown": metrics["max_drawdown"],
                "turnover": turnover,
                "failed": False,
            }
        )

    return pd.DataFrame(fold_rows)


In [18]:
cv_summary_rows = []
fold_results_map = {}

for lookback, vol_window, max_shares in product(LOOKBACKS, VOL_WINDOWS, MAX_SHARES_GRID):
    fold_df = evaluate_params(panel, train_dates, lookback, vol_window, max_shares)
    key = (lookback, vol_window, max_shares)
    fold_results_map[key] = fold_df

    cv_summary_rows.append(
        {
            "lookback": lookback,
            "vol_window": vol_window,
            "max_shares": max_shares,
            "failed_folds": int(fold_df["failed"].sum()),
            "mean_val_pnl": float(fold_df["pnl"].mean()),
            "median_val_pnl": float(fold_df["pnl"].median()),
            "mean_sharpe": float(fold_df["sharpe"].mean()),
            "mean_max_drawdown": float(fold_df["max_drawdown"].mean()),
            "mean_turnover": float(fold_df["turnover"].mean()),
        }
    )

cv_results = pd.DataFrame(cv_summary_rows)
cv_results = cv_results.sort_values(
    by=["failed_folds", "mean_val_pnl", "mean_sharpe", "mean_max_drawdown", "mean_turnover"],
    ascending=[True, False, False, False, True],
).reset_index(drop=True)

cv_results.head(15)


,lookback,vol_window,max_shares,failed_folds,mean_val_pnl,median_val_pnl,mean_sharpe,mean_max_drawdown,mean_turnover
0,120,50,100,0,6072.830,-567.25,0.562246,-0.263815,13663.2
1,120,10,100,0,5945.428,-969.26,0.333909,-0.395534,24269.0
2,120,60,100,0,5910.148,-444.43,0.540701,-0.264778,13119.0
3,120,40,100,0,5703.618,-254.10,0.438620,-0.291182,14371.0
4,120,30,100,0,5168.886,-996.02,0.290577,-0.294293,15264.8
5,20,60,100,0,4623.156,-2803.75,0.156727,-0.423615,26131.4
6,20,30,100,0,4571.448,-2532.22,0.165434,-0.450355,28272.0
7,20,10,100,0,4312.664,-4961.78,0.048056,-0.521751,37428.6
8,20,40,100,0,4052.826,-5145.06,0.091433,-0.436328,27304.6
9,120,10,50,0,3978.268,-250.74,0.544305,-0.155720,12215.2


In [19]:
best_params = tuple(cv_results.loc[0, ["lookback", "vol_window", "max_shares"]].astype(int))
best_fold_results = fold_results_map[best_params]

print(f"Best params: lookback={best_params[0]}, vol_window={best_params[1]}, max_shares={best_params[2]}")
best_fold_results


Best params: lookback=120, vol_window=50, max_shares=100


,fold,train_start,train_end,val_start,val_end,pnl,sharpe,max_drawdown,turnover,failed
0,1,2017-03-15,2017-09-15,2017-09-18,2018-03-16,-567.25,-0.177699,-0.082819,11629,False
1,2,2017-03-15,2018-03-16,2018-03-19,2018-09-13,-1180.00,-0.037206,-0.360711,15936,False
2,3,2017-03-15,2018-09-13,2018-09-14,2019-03-15,2991.82,0.691341,-0.443539,13849,False
3,4,2017-03-15,2019-03-15,2019-03-18,2019-09-12,-2896.60,-0.702119,-0.299756,13879,False
4,5,2017-03-15,2019-09-12,2019-09-13,2020-03-12,32016.18,3.036914,-0.132250,13023,False


In [20]:
best_strategy = build_tsmom_strategy(panel, *best_params)
full_port_values, full_pnl = run_backtest(panel, best_strategy["actions"])

train_end_idx = len(train_dates) - 1
test_start_idx = len(train_dates)

period_summary = pd.DataFrame(
    [
        {
            "period": "train",
            "start": pd.Timestamp(train_dates[0]).date(),
            "end": pd.Timestamp(train_dates[-1]).date(),
            "days": len(train_dates),
            **summarize_segment(full_port_values, 0, train_end_idx),
        },
        {
            "period": "test",
            "start": pd.Timestamp(test_dates[0]).date(),
            "end": pd.Timestamp(test_dates[-1]).date(),
            "days": len(test_dates),
            **summarize_segment(full_port_values, test_start_idx, len(panel) - 1),
        },
        {
            "period": "full",
            "start": panel.index[0].date(),
            "end": panel.index[-1].date(),
            "days": len(panel),
            **summarize_segment(full_port_values, 0, len(panel) - 1),
        },
    ]
)

period_summary


,period,start,end,days,pnl,sharpe,max_drawdown,start_value,end_value
0,train,2017-03-15,2020-03-12,754,22756.33,0.643801,-0.608862,25000.00,47756.33
1,test,2020-03-13,2021-03-12,252,-31538.63,-0.572680,-0.773937,47756.33,16217.70
2,full,2017-03-15,2021-03-12,1006,-8782.30,0.201718,-0.773937,25000.00,16217.70


In [21]:
signal_counts = best_strategy["signal"].stack().value_counts().sort_index().rename("count")
share_counts = best_strategy["shares"].stack().value_counts().sort_index().rename("count")
position_counts = best_strategy["positions"].stack().value_counts().sort_index().rename("count")

pd.concat(
    {
        "signal": signal_counts,
        "shares": share_counts,
        "positions": position_counts,
    },
    axis=1,
).fillna(0).astype(int)


,signal,shares,positions
-1,9916,0,0
0,3659,3659,3659
1,16605,0,0
6,0,47,0
7,0,3,0
...,...,...,...
-11,0,0,37
-10,0,0,63
-9,0,0,52
-7,0,0,3
